# TRABALHO PRÁTICO - PRÉ PROCESSAMENTO E ANÁLISE PREDITIVA

## Aluno: Luca Samuel dos Santos

## Professor: Prof. Douglas Castilho

## Matéria: Inteligência Artificial

---

## Base de Dados: Chronic Kidney Disease

## Objetivo do trabalho:

Realizar o pré-processamento e a análise preditiva sobre a base de dados de Doença Renal Crônica. Este relatório contém o passo a passo detalhado das etapas de um bom pré-processamento e de uma boa análise preditiva, descrevendo tudo o que foi pensado durante a realização do trabalho.

---

### 1. Identificação do atributo alvo (saída):

Na base de dados trabalhada, o atributo alvo é o atributo **class** podendo conter duas classes: *cdk* e *notcdk*

**Obs**: Base desbalanceada. Para uma base de 400 dados, 250 são *cdk* e 150 são *notcdk*

### 2,3. Identificação dos tipos e escala de dados dos atributos de entrada (quantitativo, qualitativo) e (nominal, ordinal, intervalar, racional)

- age (Idade) - Quantitativo, Racional: Idade em Anos
- bp (Pressão Sanguínea) - Quantitativo, Racional: Pressão em mg/Hg
- sp (Gravidade Específica) - Qualitativo, Ordinal: Dividido entre as classes [1.005,1.010,1.015,1.020,1.025]
- al (Albumina) - Qualitativo, Ordinal: Dividido entre as classes [0,1,2,3,4,5]
- su (Açúcar) - Qualitativo, Ordinal: Dividido entre as classes [0,1,2,3,4,5]
- rbc (Glóbulos Vermelhos) - Qualitativo, Nominal: Dividido entre as classes [normal,abnormal]
- pc (Células de Pus) - Qualitativo, Nominal: Dividido entre as classes [normal,abnormal]
- pcc (Aglomerados de Células de Pus) - Qualitativo, Nominal: Dividido entre as classes [present,notpresent]
- ba (Bactéria) - Qualitativo, Nominal: Dividido entre as classes [present,notpresent]
- bgr (Glicose Sanguínea Aleatória) - Quantitativo, Racional: Medido em mgs/dl
- bu (Ureia no Sangue) - Quantitativo, Racional: Medido em mgs/dl
- sc (Creatinina Sérica) - Quantitativo, Racional: Medido em mgs/dl
- sod (Sódio) - Quantitativo, Racional: Medido em mEq/L
- pot (Potássio) - Quantitativo, Racional: Medido em mEq/L
- hemo (Hemoglobina) - Quantitativo, Racional: Medido em gms
- pcv (Volume Globular/Hematócrito) - Quantitativo, Racional
- wc (Contagem de Glóbulos Brancos) - Quantitativo, Racional: cells/cumm
- rc (Contagem de Glóbulos Vermelhos) - Quantitativo, Racional: millions/cmm
- htn (Hipertensão) - Qualitativo, Nominal: Dividido entre as classes [yes,no]
- dm (Diabetes) - Qualitativo, Nominal: Dividido entre as classes [yes,no]
- cad (Doença Arterial Coronariana) - Qualitativo, Nominal: Dividido entre as classes [yes,no]
- appet (Apetite) - Qualitativo, Ordinal: Dividido entre as classes [good,poor]
- pe (Edema Podal) - Qualitativo, Nominal: Dividido entre as classes [yes,no]
- ane (Anemia) - Qualitativo, Nominal: Dividido entre as classes [yes,no]
- class (Classe/Diagnóstico) - Qualitativo, Nominal: Dividido entre as classes [cdk,notcdk]

Código para armazenamento dos atributos e dados a partir do arquivo "chronic_kidney_disease":


In [1]:
def ler_arff(caminho_arquivo):
    atributos = []
    dados = []
    lendo_dados = False

    with open(caminho_arquivo, 'r', encoding='utf-8') as arquivo:
        for linha in arquivo:
            linha = linha.strip()
            if not linha or linha.startswith('%'):
                continue
            if linha.upper().startswith('@ATTRIBUTE'):
                partes = linha.split()
                if partes[1].startswith("'") and partes[1].endswith("'"):
                    partes[1] = partes[1][1:-1]
                nome_atributo = partes[1]
                atributos.append(nome_atributo)
            elif linha.upper().startswith('@DATA'):
                lendo_dados = True
            elif lendo_dados:
                valores = [valor.strip() for valor in linha.split(',')]
                dados.append(valores)
    return atributos, dados


In [2]:
caminho = './base_de_dados/chronic_kidney_disease.arff'

try:
    atributos, dados = ler_arff(caminho)

    print("ATRIBUTOS LIDOS:")
    print(atributos)

    for dado in dados:
        for i in range(len(dado)):
            if dado[i].isdigit():
                dado[i] = int(dado[i])
            elif dado[i].replace('.', '', 1).isdigit():
                dado[i] = float(dado[i])

    print("\nPRIMEIROS 5 DADOS LIDOS:")
    for i in range(min(5, len(dados))):
        print(dados[i])

    print(f"\nTotal de atributos: {len(atributos)}")
    print(f"Total de dados: {len(dados)}")
except FileNotFoundError:
    print(f"Erro: O arquivo '{caminho}' não foi encontrado.")


ATRIBUTOS LIDOS:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'class']

PRIMEIROS 5 DADOS LIDOS:
[48, 80, 1.02, 1, 0, '?', 'normal', 'notpresent', 'notpresent', 121, 36, 1.2, '?', '?', 15.4, 44, 7800, 5.2, 'yes', 'yes', 'no', 'good', 'no', 'no', 'ckd']
[7, 50, 1.02, 4, 0, '?', 'normal', 'notpresent', 'notpresent', '?', 18, 0.8, '?', '?', 11.3, 38, 6000, '?', 'no', 'no', 'no', 'good', 'no', 'no', 'ckd']
[62, 80, 1.01, 2, 3, 'normal', 'normal', 'notpresent', 'notpresent', 423, 53, 1.8, '?', '?', 9.6, 31, 7500, '?', 'no', 'yes', 'no', 'poor', 'no', 'yes', 'ckd']
[48, 70, 1.005, 4, 0, 'normal', 'abnormal', 'present', 'notpresent', 117, 56, 3.8, 111, 2.5, 11.2, 32, 6700, 3.9, 'yes', 'no', 'no', 'poor', 'yes', 'yes', 'ckd']
[51, 80, 1.01, 2, 0, 'normal', 'normal', 'notpresent', 'notpresent', 106, 26, 1.4, '?', '?', 11.6, 35, 7300, 4.6, 'no', 'no', 'no', 'good', 'no', 'no', '

### 4,5,6. Exploracao dos dados atraves de medidas de localidade, espalhamento e distribuicao

Codigo utilizado para encontrar as medidas de localidade, adotando que para dados simbolicos, sera utilizado a moda e para atributos numericos serao utilizados media, mediana e percentil:


In [3]:
"""
0 - idade, quantitativo racional
1 - pressao arterial, quantitativo racional
9 - bgr, quantitativo racional
10 - bu, quantitativo racional
11 - sc, quantitativo racional
12 - sod, quantitativo racional
13 - pot, quantitativo racional
14 - hemo, quantitativo racional
15 - pcv, quantitativo racional
16 - wbcc, quantitativo racional
17 - rbcc, quantitativo racional
"""

atributos_quantitativos = [0, 1, 9, 10, 11, 12, 13, 14, 15, 16, 17]
percentis = {}

def calcular_outliers(dados, k25, k75):
    limite_inferior = k25 - (1.5 * (k75 - k25))
    limite_superior = k75 + (1.5 * (k75 - k25))
    outliers_inferiores = [d for d in dados if d < limite_inferior]
    outliers_superiores = [d for d in dados if d > limite_superior]
    return outliers_inferiores, outliers_superiores

def calcular_percentil(dados, percentil):
    n = len(dados)
    if n == 0: return None
    k = (n - 1) * percentil
    f = int(k)
    c = f + 1
    if c >= n: return dados[f]
    return (dados[f] + dados[c]) / 2

def dados_nao_nulos_int_float(dados, indice):
    return [dado[indice] for dado in dados if isinstance(dado[indice], (int, float))]

print("\nMedidas de Localidade dos Atributos Quantitativos:")
for indice in atributos_quantitativos:
    print(f"\nAnalise do atributo: {atributos[indice]}")
    dados_not_null = dados_nao_nulos_int_float(dados, indice)
    if dados_not_null:
        media = sum(dados_not_null) / len(dados_not_null); print(f"Media: {media:.2f}")
        mediana = sorted(dados_not_null); n = len(mediana)
        mediana = mediana[n // 2] if n % 2 == 1 else (mediana[n // 2 - 1] + mediana[n // 2]) / 2
        print(f"Mediana: {mediana:.2f}")
        dados_ord = sorted(dados_not_null)
        k25 = calcular_percentil(dados_ord, 0.25); k50 = calcular_percentil(dados_ord, 0.50); k75 = calcular_percentil(dados_ord, 0.75)
        print(f"Percentil 25: {k25:.2f}"); print(f"Percentil 50: {k50:.2f}"); print(f"Percentil 75: {k75:.2f}")
        percentis[atributos[indice]] = (k25, k50, k75)
        print(f"Medidas de Dispersao do atributo: {atributos[indice]}")
        lim_inf = k25 - 1.5*(k75-k25); lim_sup = k75 + 1.5*(k75-k25)
        print(f"Intervalo: {(dados_ord[-1]-dados_ord[0]):.2f}")
        print(f"Limite Inferior: {lim_inf:.2f}"); print(f"Limite Superior: {lim_sup:.2f}")
        out_inf, out_sup = calcular_outliers(dados_ord, k25, k75)
        print(f"Outliers abaixo: {len(out_inf)}"); print(f"Outliers acima: {len(out_sup)}")
        variancia = sum((x-media)**2 for x in dados_not_null) / (len(dados_not_null)-1)
        desvio_padrao = variancia ** 0.5
        print(f"Variancia: {variancia:.2f}"); print(f"Desvio Padrao: {desvio_padrao:.2f}")
    else:
        print("Nenhum dado numerico para", atributos[indice])



Medidas de Localidade dos Atributos Quantitativos:

Analise do atributo: age
Media: 51.48
Mediana: 55.00
Percentil 25: 42.00
Percentil 50: 55.00
Percentil 75: 64.50
Medidas de Dispersao do atributo: age
Intervalo: 88.00
Limite Inferior: 8.25
Limite Superior: 98.25
Outliers abaixo: 10
Outliers acima: 0
Variancia: 294.80
Desvio Padrao: 17.17

Analise do atributo: bp
Media: 76.47
Mediana: 80.00
Percentil 25: 70.00
Percentil 50: 80.00
Percentil 75: 80.00
Medidas de Dispersao do atributo: bp
Intervalo: 130.00
Limite Inferior: 55.00
Limite Superior: 95.00
Outliers abaixo: 5
Outliers acima: 31
Variancia: 187.24
Desvio Padrao: 13.68

Analise do atributo: bgr
Media: 148.04
Mediana: 121.00
Percentil 25: 99.00
Percentil 50: 121.00
Percentil 75: 163.00
Medidas de Dispersao do atributo: bgr
Intervalo: 468.00
Limite Inferior: 3.00
Limite Superior: 259.00
Outliers abaixo: 0
Outliers acima: 34
Variancia: 6285.59
Desvio Padrao: 79.28

Analise do atributo: bu
Media: 57.43
Mediana: 42.00
Percentil 25: 2

In [4]:
"""
2 - sg, qualitativo ordinal
3 - al, qualitativo ordinal
4 - su, qualitativo ordinal
5 - rbc, qualitativo nominal
6 - pc, qualitativo nominal
7 - pcc, qualitativo nominal
8 - ba, qualitativo nominal
18 - htn, qualitativo nominal
19 - dm, qualitativo nominal
20 - cad, qualitativo nominal
21 - appet, qualitativo ordinal
22 - pe, qualitativo nominal
23 - ane, qualitativo nominal
24 - class, qualitativo nominal
"""

atributos_qualitativos = [2, 3, 4, 5, 6, 7, 8, 18, 19, 20, 21, 22, 23, 24]
atributos_moda_percentual = {}

print("\nMedidas de Localidade dos Atributos Qualitativos:")
for indice in atributos_qualitativos:
    print(f"\nAnalise do atributo: {atributos[indice]}")
    contagem = {}
    for dado in dados:
        valor = dado[indice]
        if valor != "?": contagem[valor] = contagem.get(valor, 0) + 1
    print("Contagem de cada categoria:")
    for categoria, cont in contagem.items(): print(f"{categoria}: {cont}")
    moda = max(contagem, key=contagem.get)
    print("Moda:", moda)
    print("Percentual de cada categoria:")
    total = sum(contagem.values())
    percentuais = {c: (cont/total)*100 for c, cont in contagem.items()}
    atributos_moda_percentual[atributos[indice]] = (moda, percentuais)
    print(percentuais)



Medidas de Localidade dos Atributos Qualitativos:

Analise do atributo: sg
Contagem de cada categoria:
1.02: 106
1.01: 84
1.005: 7
1.015: 75
1.025: 81
Moda: 1.02
Percentual de cada categoria:
{1.02: 30.02832861189802, 1.01: 23.79603399433428, 1.005: 1.9830028328611897, 1.015: 21.246458923512748, 1.025: 22.946175637393768}

Analise do atributo: al
Contagem de cada categoria:
1: 44
4: 24
2: 43
3: 43
0: 199
5: 1
Moda: 0
Percentual de cada categoria:
{1: 12.429378531073446, 4: 6.779661016949152, 2: 12.146892655367232, 3: 12.146892655367232, 0: 56.21468926553672, 5: 0.2824858757062147}

Analise do atributo: su
Contagem de cada categoria:
0: 290
3: 14
4: 13
1: 13
2: 18
5: 3
Moda: 0
Percentual de cada categoria:
{0: 82.62108262108262, 3: 3.9886039886039883, 4: 3.7037037037037033, 1: 3.7037037037037033, 2: 5.128205128205128, 5: 0.8547008547008548}

Analise do atributo: rbc
Contagem de cada categoria:
normal: 201
abnormal: 47
Moda: normal
Percentual de cada categoria:
{'normal': 81.04838709677

### 7. Identificacao e separacao do conjunto de teste

Como a base e desbalanceada, 250 cdk e 150 notcdk, a divisao vai ser feita mantendo a proporcao de 62,5% e 37,5%. A divisao e feita antes mesmo de limpar os dados. A divisao da base sera de 80/20, 80% para treinamento e 20% para testes. Sendo assim, a distribuicao ficara 320 dados na base de treinamento e 80 dados na base de teste.

Codigo usado para separar o conjunto de teste:


In [5]:
from sklearn.model_selection import train_test_split

indices = list(range(len(dados)))
classes = [d[-1] for d in dados]

treino_idx, teste_idx = train_test_split(
    indices, test_size=80, stratify=classes, random_state=42
)

base_treinamento = [dados[i] for i in treino_idx]
base_teste = [dados[i] for i in teste_idx]

print(f"\nTotal de dados para treinamento: {len(base_treinamento)}")
print(f"Total de dados para teste: {len(base_teste)}")

print("\nCdk e notcdk na base de teste:")
print(f"ckd: {sum(1 for d in base_teste if d[-1] == 'ckd')}")
print(f"notckd: {sum(1 for d in base_teste if d[-1] == 'notckd')}")

print("\nCdk e notcdk na base de treinamento:")
print(f"ckd: {sum(1 for d in base_treinamento if d[-1] == 'ckd')}")
print(f"notckd: {sum(1 for d in base_treinamento if d[-1] == 'notckd')}")

# Salvar classe majoritaria original para o baseline (antes do balanceamento)
classes_orig = {}
for d in base_treinamento:
    c = d[-1]; classes_orig[c] = classes_orig.get(c, 0) + 1
classe_majoritaria_orig = max(classes_orig, key=classes_orig.get)



Total de dados para treinamento: 320
Total de dados para teste: 80

Cdk e notcdk na base de teste:
ckd: 50
notckd: 30

Cdk e notcdk na base de treinamento:
ckd: 200
notckd: 120


### 8,9. Identificacao e eliminacao de atributos e exemplos nao necessarios

Como a base de dados nao tem atributos facilmente detectaveis que sao dispensaveis, a tecnica utilizada foi a medida de variancia e frequencia com que as classes aparecem.

De acordo com os dados obtidos, escolhi utilizar os seguintes atributos:
- Quantitativos: age, bp, bgr, bu, sod, wbcc
- Qualitativos: sg, al, htn, dm

Codigo que faz a identificacao e a eliminacao desses atributos


In [6]:
atributos_utilizados = ["age", "bp", "sg", "al", "bgr", "bu", "sod", "wbcc", "htn", "dm", "class"]
col_atributos_utilizados = []
for i in range(len(atributos)):
    if atributos[i] in atributos_utilizados:
        col_atributos_utilizados.append(atributos.index(atributos[i]))


### 10. Analise e aplicacao de tecnicas de amostragem de dados

Nao ha necessidade de amostragem, dado que a base contem apenas 400 linhas.

### 11. Desbalanceamento

Como a classe majoritaria tem 62,5% dos dados, usarei acrescimo a classe minoritaria para balancear. A base e muito pequena para eliminar dados.

NOTA: O oversampling e feito APOS a limpeza dos dados para evitar contaminacao das duplicatas com tratamento de outliers/omissos.


In [7]:
# O oversampling sera feito APOS toda a limpeza e normalizacao
# (ver celula dedicada mais abaixo)


### 12. Limpeza de dados

a. Outliers
b. Dados inconsistentes
c. Dados reduntantes
d. Dados incompletos

Para dados incompletos usarei heuristica de preenchimento. Para outliers, substituirei pelo limite mais proximo.


In [8]:
outliers_por_atributo = {}
atributos_quantitativos_utilizados = [0, 1, 9, 10, 12, 16]
atributos_qualitativos_utilizados = [2, 3, 18, 19]

for indice in atributos_quantitativos_utilizados:
    dados_nao_nulos_ordenados = dados_nao_nulos_int_float(base_treinamento, indice)
    dados_nao_nulos_ordenados = sorted(dados_nao_nulos_ordenados)
    k25 = calcular_percentil(dados_nao_nulos_ordenados, 0.25)
    k75 = calcular_percentil(dados_nao_nulos_ordenados, 0.75)
    limite_inferior = k25 - (1.5 * (k75 - k25))
    limite_superior = k75 + (1.5 * (k75 - k25))
    out_inf, out_sup = calcular_outliers(dados_nao_nulos_ordenados, k25, k75)
    print(f"\nAtributo: {atributos[indice]}")
    print(f"Outliers abaixo: {len(out_inf)}, acima: {len(out_sup)}")
    for dado in base_treinamento:
        if isinstance(dado[indice], (int, float)):
            if dado[indice] < limite_inferior: dado[indice] = limite_inferior
            elif dado[indice] > limite_superior: dado[indice] = limite_superior
    btreino_vals = dados_nao_nulos_int_float(base_treinamento, indice)
    out_inf2, out_sup2 = calcular_outliers(btreino_vals, k25, k75)
    print(f"Apos tratamento: outliers abaixo: {len(out_inf2)}, acima: {len(out_sup2)}")
    subst = 0
    for dado in base_treinamento:
        if dado[indice] == "?":
            dado[indice] = dados_nao_nulos_ordenados[len(dados_nao_nulos_ordenados)//2]; subst += 1
    print(f"Substituidos: {subst}")

for indice in atributos_qualitativos_utilizados:
    contagem = {}
    for dado in base_treinamento:
        valor = dado[indice]
        if valor != "?": contagem[valor] = contagem.get(valor, 0) + 1
    moda = max(contagem, key=contagem.get)
    subst = 0
    for dado in base_treinamento:
        if dado[indice] == "?": dado[indice] = moda; subst += 1
    print(f"\nAtributo: {atributos[indice]}")
    print(f"Moda: {moda}, substituidos: {subst}")



Atributo: age
Outliers abaixo: 5, acima: 0
Apos tratamento: outliers abaixo: 0, acima: 0
Substituidos: 5

Atributo: bp
Outliers abaixo: 4, acima: 25
Apos tratamento: outliers abaixo: 0, acima: 0
Substituidos: 10

Atributo: bgr
Outliers abaixo: 0, acima: 35
Apos tratamento: outliers abaixo: 0, acima: 0
Substituidos: 38

Atributo: bu
Outliers abaixo: 0, acima: 30
Apos tratamento: outliers abaixo: 0, acima: 0
Substituidos: 17

Atributo: sod
Outliers abaixo: 11, acima: 1
Apos tratamento: outliers abaixo: 0, acima: 0
Substituidos: 72

Atributo: wbcc
Outliers abaixo: 0, acima: 9
Apos tratamento: outliers abaixo: 0, acima: 0
Substituidos: 89

Atributo: sg
Moda: 1.02, substituidos: 33

Atributo: al
Moda: 0, substituidos: 34

Atributo: htn
Moda: no, substituidos: 0

Atributo: dm
Moda: no, substituidos: 0


### 13. Identificacao e conversao dos tipos de dados

a. Conversao de tipos
b. Normalizacao dos dados


### Aplicacao do filtro de atributos selecionados nas bases de treinamento e teste


In [9]:
indices_utilizados = [atributos.index(a) for a in atributos_utilizados]

def filtrar_colunas(dados_base, indices):
    return [[dado[i] for i in indices] for dado in dados_base]

base_treinamento = filtrar_colunas(base_treinamento, indices_utilizados)
base_teste = filtrar_colunas(base_teste, indices_utilizados)
atributos_finais = atributos_utilizados

print("Atributos finais:", atributos_finais)
print("Treinamento: {}x{}".format(len(base_treinamento), len(base_treinamento[0])))
print("Teste: {}x{}".format(len(base_teste), len(base_teste[0])))


Atributos finais: ['age', 'bp', 'sg', 'al', 'bgr', 'bu', 'sod', 'wbcc', 'htn', 'dm', 'class']
Treinamento: 320x11
Teste: 80x11


### Limpeza da base de teste e correcao de dados inconsistentes

Utilizando as proprias estatisticas da base de treinamento (mediana para numericos, moda para categoricos) para preencher valores '?' na base de teste, evitando vazamento de dados. Outliers usam limites da base de treinamento.


In [10]:
atributos_quant_utilizados = [0, 1, 4, 5, 6, 7]
atributos_qual_utilizados = [2, 3, 8, 9]

def calc_mediana(vals):
    vals = sorted(vals); n = len(vals)
    if n == 0: return 0
    return vals[n//2] if n%2==1 else (vals[n//2-1]+vals[n//2])/2

def calc_moda(vals):
    cont = {}
    for v in vals: cont[v] = cont.get(v, 0) + 1
    return max(cont, key=cont.get)

# Estatisticas da base de treinamento (para outliers e inconsistentes)
medians_treino = {}; limits = {}
for idx in atributos_quant_utilizados:
    vals = [d[idx] for d in base_treinamento if isinstance(d[idx], (int, float))]
    vals.sort(); medians_treino[idx] = calc_mediana(vals)
    n = len(vals); k25 = vals[int((n-1)*0.25)]; k75 = vals[int((n-1)*0.75)]
    limits[idx] = (k25-1.5*(k75-k25), k75+1.5*(k75-k25))

modes_treino = {}
for idx in atributos_qual_utilizados:
    vals = [d[idx] for d in base_treinamento if d[idx] != "?"]
    modes_treino[idx] = calc_moda(vals)

print("Medians treino:", medians_treino)
print("Modas treino  :", modes_treino)

# Preencher "?" na base de treinamento com estatisticas da propria base
for dado in base_treinamento:
    for idx in atributos_quant_utilizados:
        if not isinstance(dado[idx], (int, float)): dado[idx] = medians_treino[idx]
    for idx in atributos_qual_utilizados:
        if dado[idx] not in ["yes","no","ckd","notckd"]:
            if dado[idx] == "?" or dado[idx] == "": dado[idx] = modes_treino[idx]

# Preencher "?" na base de teste com estatisticas do treino (evita vazamento)
for dado in base_teste:
    for idx in atributos_quant_utilizados:
        if not isinstance(dado[idx], (int, float)): dado[idx] = medians_treino[idx]
    for idx in atributos_qual_utilizados:
        if dado[idx] not in ["yes","no","ckd","notckd"]:
            if dado[idx] == "?" or dado[idx] == "": dado[idx] = modes_treino[idx]

# Tratamento de outliers na base de teste (usa limites do treino)
for dado in base_teste:
    for idx in atributos_quant_utilizados:
        if isinstance(dado[idx], (int, float)):
            lim_inf, lim_sup = limits[idx]
            if dado[idx] < lim_inf: dado[idx] = lim_inf
            elif dado[idx] > lim_sup: dado[idx] = lim_sup
    for idx in atributos_qual_utilizados:
        if dado[idx] == "?": dado[idx] = modes_treino[idx]

print("\nLimpeza concluida.")

def remover_duplicados(base):
    vistos = []
    for d in base:
        if tuple(d) not in vistos: vistos.append(tuple(d))
    return [list(t) for t in vistos]

antes = len(base_teste)
base_teste = remover_duplicados(base_teste)
print("Duplicadas removidas teste: {}".format(antes - len(base_teste)))
print("Teste: {} linhas".format(len(base_teste)))
print("Treino: {} linhas".format(len(base_treinamento)))


Medians treino: {0: 54.0, 1: 80.0, 4: 120.0, 5: 41.0, 6: 138.0, 7: 8000.0}
Modas treino  : {2: 1.02, 3: 0, 8: 'no', 9: 'no'}

Limpeza concluida.
Duplicadas removidas teste: 0
Teste: 80 linhas
Treino: 320 linhas


### 13.a. Conversao de tipos

- htn: yes -> 1, no -> 0
- dm: yes -> 1, no -> 0
- class: ckd -> 1, notckd -> 0


In [11]:
def conv_bin(v): return 1 if v=="yes" else 0
def conv_class(v): return 1 if v=="ckd" else 0

for d in base_treinamento:
    d[8]=conv_bin(d[8]); d[9]=conv_bin(d[9]); d[10]=conv_class(d[10])
for d in base_teste:
    d[8]=conv_bin(d[8]); d[9]=conv_bin(d[9]); d[10]=conv_class(d[10])

print("Conversao concluida.")
print("Treino - ckd: {}, notckd: {}".format(sum(1 for d in base_treinamento if d[10]==1), sum(1 for d in base_treinamento if d[10]==0)))
print("Teste - ckd: {}, notckd: {}".format(sum(1 for d in base_teste if d[10]==1), sum(1 for d in base_teste if d[10]==0)))


Conversao concluida.
Treino - ckd: 200, notckd: 120
Teste - ckd: 50, notckd: 30


### 13.b. Normalizacao (Min-Max)


In [12]:
atributos_normalizar = [0, 1, 2, 3, 4, 5, 6, 7]
minimos={}; maximos={}
for idx in atributos_normalizar:
    vals = [d[idx] for d in base_treinamento if isinstance(d[idx], (int, float))]
    minimos[idx]=min(vals); maximos[idx]=max(vals)

def normalizar(v, mi, ma): return 0.5 if ma==mi else (v-mi)/(ma-mi)

for base in [base_treinamento, base_teste]:
    for d in base:
        for idx in atributos_normalizar:
            if isinstance(d[idx], (int, float)): d[idx] = normalizar(d[idx], minimos[idx], maximos[idx])

print("Normalizacao concluida.")


Normalizacao concluida.


### 14. Reducao de Dimensionalidade (filtro por correlacao)


In [13]:
def correlacao_pearson(x, y):
    n=len(x); mx=sum(x)/n; my=sum(y)/n
    varx=sum((xi-mx)**2 for xi in x); vary=sum((yi-my)**2 for yi in y)
    if varx==0 or vary==0: return 0
    return sum((x[i]-mx)*(y[i]-my) for i in range(n))/((varx*vary)**0.5)

n_attr = len(atributos_finais)-1; colunas={}
for i in range(n_attr): colunas[i] = [d[i] for d in base_treinamento]

print("Matriz de correlacao:"); cab=" "*10
for j in range(n_attr): cab += "{:>10}".format(atributos_finais[j][:8])
print(cab)
for i in range(n_attr):
    linha = "{:>10}".format(atributos_finais[i][:8])
    for j in range(n_attr): linha += "{:>10.4f}".format(correlacao_pearson(colunas[i], colunas[j]))
    print(linha)

target_treino = [d[10] for d in base_treinamento]
print("\nCorrelacao com target:")
for i in range(n_attr):
    print("{}: {:.4f}".format(atributos_finais[i], correlacao_pearson(colunas[i], target_treino)))

atributos_manter = list(range(n_attr)); removidos=[]
for i in range(n_attr):
    if i not in atributos_manter: continue
    for j in range(i+1, n_attr):
        if j not in atributos_manter: continue
        corr = abs(correlacao_pearson(colunas[i], colunas[j]))
        if corr > 0.85:
            ci = abs(correlacao_pearson(colunas[i], target_treino))
            cj = abs(correlacao_pearson(colunas[j], target_treino))
            if ci < cj:
                if i not in [r[0] for r in removidos]: removidos.append((i, atributos_finais[i], corr)); atributos_manter.remove(i)
            else:
                if j not in [r[0] for r in removidos]: removidos.append((j, atributos_finais[j], corr)); atributos_manter.remove(j)

if removidos:
    print("\nRemovidos:")
    for idx,nome,corr in removidos: print("  - {} (corr: {:.4f})".format(nome, corr))
else: print("\nNenhum removido.")

print("\nMantidos:")
for idx in sorted(atributos_manter): print("  - {}".format(atributos_finais[idx]))

indices_reduzidos = sorted(atributos_manter)+[10]
base_treinamento = [[d[i] for i in indices_reduzidos] for d in base_treinamento]
base_teste = [[d[i] for i in indices_reduzidos] for d in base_teste]
atributos_finais = [atributos_finais[i] for i in sorted(atributos_manter)] + ["class"]

print("\nTreinamento: {}x{}".format(len(base_treinamento), len(base_treinamento[0])))
print("Teste: {}x{}".format(len(base_teste), len(base_teste[0])))
print("Atributos:", atributos_finais)


Matriz de correlacao:
                 age        bp        sg        al       bgr        bu       sod      wbcc       htn        dm
       age    1.0000    0.1880   -0.1736    0.0866    0.2434    0.2726   -0.0840    0.1055    0.3676    0.3366
        bp    0.1880    1.0000   -0.2056    0.1702    0.1770    0.2019   -0.1244    0.0191    0.2733    0.2499
        sg   -0.1736   -0.2056    1.0000   -0.4706   -0.3398   -0.2633    0.3224   -0.1969   -0.3071   -0.3461
        al    0.0866    0.1702   -0.4706    1.0000    0.2897    0.3832   -0.3605    0.2029    0.4046    0.3147
       bgr    0.2434    0.1770   -0.3398    0.2897    1.0000    0.1872   -0.2787    0.1417    0.3620    0.5282
        bu    0.2726    0.2019   -0.2633    0.3832    0.1872    1.0000   -0.3826    0.0753    0.5101    0.3975
       sod   -0.0840   -0.1244    0.3224   -0.3605   -0.2787   -0.3826    1.0000   -0.0215   -0.3963   -0.3528
      wbcc    0.1055    0.0191   -0.1969    0.2029    0.1417    0.0753   -0.0215    1.0000

In [14]:
### Oversampling (realizado apos toda limpeza/normalizacao)

import copy

# Salvar copia ANTES do oversampling para uso na validacao cruzada
base_treinamento_original = copy.deepcopy(base_treinamento)
print(f"Base original salva: {len(base_treinamento_original)} amostras")

# Oversampling: duplicar classe minoritaria (notckd=0) para balancear
qckd = sum(1 for d in base_treinamento if d[-1] == 1)
qnot = sum(1 for d in base_treinamento if d[-1] == 0)

print(f"\nAntes oversampling - ckd: {qckd}, notckd: {qnot}")

if qckd < qnot:
    falta = qnot - qckd
    for d in base_treinamento:
        if d[-1] == 1 and falta > 0:
            falta -= 1; base_treinamento.append(copy.deepcopy(d))
elif qckd > qnot:
    falta = qckd - qnot
    for d in base_treinamento:
        if d[-1] == 0 and falta > 0:
            falta -= 1; base_treinamento.append(copy.deepcopy(d))

print(f"Apos - CKD: {sum(1 for d in base_treinamento if d[-1] == 1)}")
print(f"       notCKD: {sum(1 for d in base_treinamento if d[-1] == 0)}")


Base original salva: 320 amostras

Antes oversampling - ckd: 200, notckd: 120
Apos - CKD: 200
       notCKD: 200


# FASE 2: ANALISE PREDITIVA


### 1 & 2. Metricas (Matriz de Confusao manual)

VP: ckd->ckd | VN: notckd->notckd | FP: notckd->ckd | FN: ckd->notckd


In [15]:
def calc_matriz_confusao(reais, previstos, pos=1, neg=0):
    vp=vn=fp=fn=0
    for i in range(len(reais)):
        if reais[i]==pos and previstos[i]==pos: vp+=1
        elif reais[i]==neg and previstos[i]==neg: vn+=1
        elif reais[i]==neg and previstos[i]==pos: fp+=1
        elif reais[i]==pos and previstos[i]==neg: fn+=1
    return vp, vn, fp, fn

def calc_metricas(vp, vn, fp, fn):
    total=vp+vn+fp+fn
    acuracia = (vp+vn)/total if total>0 else 0
    precisao = vp/(vp+fp) if (vp+fp)>0 else 0
    recall = vp/(vp+fn) if (vp+fn)>0 else 0
    especificidade = vn/(vn+fp) if (vn+fp)>0 else 0
    f1 = 2*precisao*recall/(precisao+recall) if (precisao+recall)>0 else 0
    return {
        "Acuracia": acuracia,
        "Precisao": precisao,
        "Recall": recall,
        "Especificidade": especificidade,
        "F1-Score": f1
    }

def exibir(nome, vp, vn, fp, fn):
    print("=== {} ===".format(nome))
    print("           Prev POS  Prev NEG")
    print("Real POS   {:>9} {:>9}".format(vp, fn))
    print("Real NEG   {:>9} {:>9}".format(fp, vn))
    print()
    met = calc_metricas(vp, vn, fp, fn)
    for k,v in met.items(): print("{}: {:.4f}".format(k, v))
    return met


In [16]:
# Separar dados para modelos
# DADOS ORIGINAIS (sem oversampling) para validacao cruzada
# DADOS BALANCEADOS (com oversampling) para treino final

# --- Dados originais (para CV) ---
X_treino_original = [d[:-1] for d in base_treinamento_original]
y_treino_original = [d[-1] for d in base_treinamento_original]

# --- Dados balanceados (para treino final) ---
X_treino = [d[:-1] for d in base_treinamento]
y_treino = [d[-1] for d in base_treinamento]

X_teste = [d[:-1] for d in base_teste]
y_teste = [d[-1] for d in base_teste]

print("--- Dados para CV (originais, sem oversampling) ---")
print("X_treino_original: {}x{}".format(len(X_treino_original), len(X_treino_original[0]) if X_treino_original else 0))
print("y_treino_original: {} amostras".format(len(y_treino_original)))
print("  ckd: {}, notckd: {}".format(sum(y_treino_original), len(y_treino_original)-sum(y_treino_original)))
print()
print("--- Dados para Treino Final (balanceados) ---")
print("X_treino: {}x{}".format(len(X_treino), len(X_treino[0])))
print("y_treino: {} amostras".format(len(y_treino)))
print("  ckd: {}, notckd: {}".format(sum(y_treino), len(y_treino)-sum(y_treino)))
print()
print("--- Dados de Teste ---")
print("X_teste: {}x{}".format(len(X_teste), len(X_teste[0])))
print("y_teste: {} amostras".format(len(y_teste)))
print("  ckd: {}, notckd: {}".format(sum(y_teste), len(y_teste)-sum(y_teste)))


--- Dados para CV (originais, sem oversampling) ---
X_treino_original: 320x10
y_treino_original: 320 amostras
  ckd: 200, notckd: 120

--- Dados para Treino Final (balanceados) ---
X_treino: 400x10
y_treino: 400 amostras
  ckd: 200, notckd: 200

--- Dados de Teste ---
X_teste: 80x10
y_teste: 80 amostras
  ckd: 50, notckd: 30


### 1. Tecnica de Validacao: Leave-One-Out (LOO)

LOO sera usada para selecionar os hiperparametros de cada modelo usando APENAS a base de treinamento ORIGINAL (sem oversampling). Apos encontrar os melhores parametros, treina-se o modelo final em TODA a base de treinamento (com oversampling) e avalia-se na base de teste.

Nota: O LOO/CV e feito sobre os dados ORIGINAIS (320 amostras) para evitar que as duplicatas do oversampling contaminem a validacao cruzada.


In [17]:
def leave_one_out(X, y, modelo_fn, **params):
    n = len(X)
    vp_total = vn_total = fp_total = fn_total = 0
    for i in range(n):
        X_train_loo = [X[j] for j in range(n) if j != i]
        y_train_loo = [y[j] for j in range(n) if j != i]
        X_test_loo = [X[i]]
        y_test_loo = [y[i]]
        y_pred_loo = modelo_fn(X_train_loo, y_train_loo, X_test_loo, **params)
        vp, vn, fp, fn = calc_matriz_confusao(y_test_loo, y_pred_loo)
        vp_total += vp; vn_total += vn; fp_total += fp; fn_total += fn
    return vp_total, vn_total, fp_total, fn_total

def loo_acuracia(X, y, modelo_fn, **params):
    vp, vn, fp, fn = leave_one_out(X, y, modelo_fn, **params)
    met = calc_metricas(vp, vn, fp, fn)
    return met["Acuracia"]

print("Funcao LOO implementada.")
print("Executando LOO com {} amostras de treinamento...".format(len(X_treino_original)))


Funcao LOO implementada.
Executando LOO com 320 amostras de treinamento...


### 3. Baseline (Classificador Majoritario)


In [18]:
classe_bl = 1 if classe_majoritaria_orig == "ckd" else 0
print("Classe majoritaria original:", classe_majoritaria_orig, "->", classe_bl, "no teste")

y_prev = [classe_bl] * len(y_teste)
vp, vn, fp, fn = calc_matriz_confusao(y_teste, y_prev)
metricas_baseline = exibir("BASELINE", vp, vn, fp, fn)


Classe majoritaria original: ckd -> 1 no teste
=== BASELINE ===
           Prev POS  Prev NEG
Real POS          50         0
Real NEG          30         0

Acuracia: 0.6250
Precisao: 0.6250
Recall: 1.0000
Especificidade: 0.0000
F1-Score: 0.7692


### 4. KNN (sklearn) - LOO para selecionar k (usando dados ORIGINAIS)


In [19]:
from sklearn.neighbors import KNeighborsClassifier

def knn_predict(X_train, y_train, X_test, **kw):
    k = kw.get("k", 1)
    knn = KNeighborsClassifier(n_neighbors=k, metric="euclidean")
    knn.fit(X_train, y_train)
    return knn.predict(X_test)

melhor_k = 1; melhor_ac = 0
print("\n--- LOO para KNN (dados originais, 320 amostras) ---")
for k in range(1, 21):
    ac = loo_acuracia(X_treino_original, y_treino_original, knn_predict, k=k)
    if ac > melhor_ac: melhor_ac = ac; melhor_k = k
    print(f"LOO k={k:2d} - Acuracia: {ac:.4f}")
print(f"\nMelhor k via LOO: {melhor_k} (Acuracia LOO: {melhor_ac:.4f})")

knn_final = KNeighborsClassifier(n_neighbors=melhor_k, metric="euclidean")
knn_final.fit(X_treino, y_treino)
yp = knn_final.predict(X_teste)
vp, vn, fp, fn = calc_matriz_confusao(y_teste, yp)
metricas_knn = exibir("KNN (k={}) - TESTE FINAL".format(melhor_k), vp, vn, fp, fn)



--- LOO para KNN (dados originais, 320 amostras) ---
LOO k= 1 - Acuracia: 0.9500
LOO k= 2 - Acuracia: 0.9469
LOO k= 3 - Acuracia: 0.9500
LOO k= 4 - Acuracia: 0.9469
LOO k= 5 - Acuracia: 0.9531
LOO k= 6 - Acuracia: 0.9469
LOO k= 7 - Acuracia: 0.9500
LOO k= 8 - Acuracia: 0.9469
LOO k= 9 - Acuracia: 0.9469
LOO k=10 - Acuracia: 0.9437
LOO k=11 - Acuracia: 0.9437
LOO k=12 - Acuracia: 0.9437
LOO k=13 - Acuracia: 0.9437
LOO k=14 - Acuracia: 0.9406
LOO k=15 - Acuracia: 0.9406
LOO k=16 - Acuracia: 0.9313
LOO k=17 - Acuracia: 0.9406
LOO k=18 - Acuracia: 0.9344
LOO k=19 - Acuracia: 0.9375
LOO k=20 - Acuracia: 0.9313

Melhor k via LOO: 5 (Acuracia LOO: 0.9531)
=== KNN (k=5) - TESTE FINAL ===
           Prev POS  Prev NEG
Real POS          47         3
Real NEG           0        30

Acuracia: 0.9625
Precisao: 1.0000
Recall: 0.9400
Especificidade: 1.0000
F1-Score: 0.9691


### 5. Arvore de Decisao C4.5 - 10-fold CV para selecionar profundidade (usando dados ORIGINAIS)


In [20]:
import math, random

class DecisionTreeC45:
    def __init__(self, max_depth=None, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.tree = None

    def fit(self, X, y):
        n = len(X)
        n_attrs = len(X[0]) if n > 0 else 0
        self.continuous_attrs = set()
        for j in range(n_attrs):
            vals = set(X[i][j] for i in range(n))
            if len(vals) > 10:
                self.continuous_attrs.add(j)
        self.tree = self._build_tree(list(range(n)), X, y, 0, set(range(n_attrs)))

    def _entropy(self, y, indices):
        if not indices:
            return 0.0
        counts = {}
        for i in indices:
            counts[y[i]] = counts.get(y[i], 0) + 1
        total = len(indices)
        ent = 0.0
        for c in counts.values():
            p = c / total
            if p > 0:
                ent -= p * math.log2(p)
        return ent

    def _gain_ratio(self, y, indices, subsets):
        total = len(indices)
        s_ent = self._entropy(y, indices)
        remainder = 0.0
        split_info = 0.0
        for s in subsets:
            if not s:
                continue
            frac = len(s) / total
            remainder += frac * self._entropy(y, s)
            split_info -= frac * math.log2(frac)
        gain = s_ent - remainder
        if split_info == 0:
            return 0.0
        return gain / split_info

    def _best_continuous_split(self, X, y, indices, attr):
        sorted_idx = sorted(indices, key=lambda i: X[i][attr])
        vals = sorted(set(X[i][attr] for i in sorted_idx))
        if len(vals) <= 1:
            return None, 0.0
        best_gr = 0.0
        best_t = None
        for k in range(len(vals) - 1):
            t = (vals[k] + vals[k + 1]) / 2.0
            left = [i for i in sorted_idx if X[i][attr] <= t]
            right = [i for i in sorted_idx if X[i][attr] > t]
            if len(left) < self.min_samples_leaf or len(right) < self.min_samples_leaf:
                continue
            gr = self._gain_ratio(y, indices, [left, right])
            if gr > best_gr:
                best_gr = gr
                best_t = t
        return best_t, best_gr

    def _best_discrete_gr(self, X, y, indices, attr):
        subsets = {}
        for i in indices:
            v = X[i][attr]
            subsets.setdefault(v, []).append(i)
        subsets_list = list(subsets.values())
        if any(len(s) < self.min_samples_leaf for s in subsets_list):
            return 0.0
        return self._gain_ratio(y, indices, subsets_list)

    def _majority(self, y, indices):
        counts = {}
        for i in indices:
            counts[y[i]] = counts.get(y[i], 0) + 1
        return max(counts, key=counts.get)

    def _build_tree(self, indices, X, y, depth, avail):
        n = len(indices)
        if n == 0:
            return {"type": "leaf", "class": 0}
        classes = set(y[i] for i in indices)
        if len(classes) == 1:
            return {"type": "leaf", "class": classes.pop()}
        if self.max_depth is not None and depth >= self.max_depth:
            return {"type": "leaf", "class": self._majority(y, indices)}
        if not avail:
            return {"type": "leaf", "class": self._majority(y, indices)}

        best_attr = None
        best_gr = 0.0
        best_t = None
        best_type = None
        for attr in avail:
            if attr in self.continuous_attrs:
                t, gr = self._best_continuous_split(X, y, indices, attr)
                if gr > best_gr:
                    best_gr = gr
                    best_attr = attr
                    best_t = t
                    best_type = "cont"
            else:
                gr = self._best_discrete_gr(X, y, indices, attr)
                if gr > best_gr:
                    best_gr = gr
                    best_attr = attr
                    best_t = None
                    best_type = "disc"

        if best_attr is None or best_gr <= 1e-10:
            return {"type": "leaf", "class": self._majority(y, indices)}

        if best_type == "cont":
            left = [i for i in indices if X[i][best_attr] <= best_t]
            right = [i for i in indices if X[i][best_attr] > best_t]
            return {
                "type": "node", "attr": best_attr, "threshold": best_t,
                "left": self._build_tree(left, X, y, depth + 1, avail),
                "right": self._build_tree(right, X, y, depth + 1, avail)
            }
        else:
            parts = {}
            for i in indices:
                v = X[i][best_attr]
                parts.setdefault(v, []).append(i)
            remaining = set(avail)
            remaining.discard(best_attr)
            branches = {}
            for v, s in parts.items():
                branches[v] = self._build_tree(s, X, y, depth + 1, remaining)
            return {
                "type": "node", "attr": best_attr, "branches": branches,
                "default": {"type": "leaf", "class": self._majority(y, indices)}
            }
    def predict(self, X):
        return [self._predict_one(x, self.tree) for x in X]

    def _predict_one(self, x, node):
        if node["type"] == "leaf":
            return node["class"]
        if "threshold" in node:
            if x[node["attr"]] <= node["threshold"]:
                return self._predict_one(x, node["left"])
            else:
                return self._predict_one(x, node["right"])
        return self._predict_one(x, node["branches"].get(x[node["attr"]], node["default"]))

def tree_predict(X_train, y_train, X_test, **kw):
    prof = kw.get("prof", 1)
    arv = DecisionTreeC45(max_depth=prof, min_samples_leaf=5)
    arv.fit(X_train, y_train)
    return arv.predict(X_test)

def kfold_acuracia(X, y, modelo_fn, n_folds=10, **params):
    n = len(X)
    indices = list(range(n))
    random.seed(42)
    random.shuffle(indices)
    fold_size = n // n_folds
    vp_tot = vn_tot = fp_tot = fn_tot = 0
    for k in range(n_folds):
        test_idx = indices[k * fold_size:(k + 1) * fold_size]
        train_idx = [i for i in indices if i not in test_idx]
        X_train_k = [X[i] for i in train_idx]
        y_train_k = [y[i] for i in train_idx]
        X_test_k = [X[i] for i in test_idx]
        y_test_k = [y[i] for i in test_idx]
        y_pred_k = modelo_fn(X_train_k, y_train_k, X_test_k, **params)
        vp, vn, fp, fn = calc_matriz_confusao(y_test_k, y_pred_k)
        vp_tot += vp; vn_tot += vn; fp_tot += fp; fn_tot += fn
    met = calc_metricas(vp_tot, vn_tot, fp_tot, fn_tot)
    return met["Acuracia"]

melhor_prof = 1; melhor_ac = 0
print("\n--- 10-fold CV para Arvore C4.5 (dados originais, 320 amostras) ---")
for prof in range(1, 21):
    ac = kfold_acuracia(X_treino_original, y_treino_original, tree_predict, n_folds=10, prof=prof)
    if ac > melhor_ac: melhor_ac = ac; melhor_prof = prof
    print(f"10-fold prof={prof:2d} - Acuracia: {ac:.4f}")
print(f"\nMelhor profundidade via 10-fold CV: {melhor_prof} (Acuracia: {melhor_ac:.4f})")

arv_final = DecisionTreeC45(max_depth=melhor_prof, min_samples_leaf=5)
arv_final.fit(X_treino, y_treino)
yp = arv_final.predict(X_teste)
vp, vn, fp, fn = calc_matriz_confusao(y_teste, yp)
metricas_arvore = exibir("C4.5 (prof={}) - TESTE FINAL".format(melhor_prof), vp, vn, fp, fn)



--- 10-fold CV para Arvore C4.5 (dados originais, 320 amostras) ---
10-fold prof= 1 - Acuracia: 0.7219
10-fold prof= 2 - Acuracia: 0.7937
10-fold prof= 3 - Acuracia: 0.8750
10-fold prof= 4 - Acuracia: 0.8969
10-fold prof= 5 - Acuracia: 0.9219
10-fold prof= 6 - Acuracia: 0.9219
10-fold prof= 7 - Acuracia: 0.9219
10-fold prof= 8 - Acuracia: 0.9156
10-fold prof= 9 - Acuracia: 0.9375
10-fold prof=10 - Acuracia: 0.9344
10-fold prof=11 - Acuracia: 0.9281
10-fold prof=12 - Acuracia: 0.9281
10-fold prof=13 - Acuracia: 0.9281
10-fold prof=14 - Acuracia: 0.9281
10-fold prof=15 - Acuracia: 0.9281
10-fold prof=16 - Acuracia: 0.9281
10-fold prof=17 - Acuracia: 0.9281
10-fold prof=18 - Acuracia: 0.9281
10-fold prof=19 - Acuracia: 0.9281
10-fold prof=20 - Acuracia: 0.9281

Melhor profundidade via 10-fold CV: 9 (Acuracia: 0.9375)
=== C4.5 (prof=9) - TESTE FINAL ===
           Prev POS  Prev NEG
Real POS          48         2
Real NEG           0        30

Acuracia: 0.9750
Precisao: 1.0000
Recall: 0.

### 6. MLP - Rede Neural (sklearn) - LOO com subconjunto (50 amostras dos dados ORIGINAIS)

LOO completo com 320 amostras para MLP seria extremamente lento (320 execucoes por configuracao). Usamos as primeiras 50 amostras dos dados ORIGINAIS para selecao da arquitetura, depois treinamos com TODOS os dados (balanceados).


In [21]:
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)
from sklearn.neural_network import MLPClassifier

# Amostra estratificada dos dados ORIGINAIS: 25 ckd + 25 notckd para LOO
import random
random.seed(42)
ckd_idx = [i for i, v in enumerate(y_treino_original) if v == 1]
notckd_idx = [i for i, v in enumerate(y_treino_original) if v == 0]
loo_idx = random.sample(ckd_idx, 25) + random.sample(notckd_idx, 25)
random.shuffle(loo_idx)
X_loo = [X_treino_original[i] for i in loo_idx]
y_loo = [y_treino_original[i] for i in loo_idx]

def mlp_predict(X_train, y_train, X_test, **kw):
    cfg = kw.get("cfg")
    mlp = MLPClassifier(hidden_layer_sizes=cfg, activation="relu", solver="adam", max_iter=2000, random_state=42, learning_rate_init=0.01)
    mlp.fit(X_train, y_train)
    return mlp.predict(X_test)

configs = [(5,), (10,), (15,), (20,), (5, 3), (10, 5)]
melhor_ac = 0; melhor_cfg = None
print("\n--- LOO (50 amostras originais) para MLP ---")
for cfg in configs:
    ac = loo_acuracia(X_loo, y_loo, mlp_predict, cfg=cfg)
    if ac > melhor_ac: melhor_ac = ac; melhor_cfg = cfg
    print("LOO {} - Acuracia: {:.4f}".format(str(cfg), ac))
print(f"\nMelhor arquitetura via LOO: {melhor_cfg} (Acuracia LOO: {melhor_ac:.4f})")

print("\n--- Treinando MLP final com todas as {} amostras (balanceadas) ---".format(len(X_treino)))
mlp_final = MLPClassifier(hidden_layer_sizes=melhor_cfg, activation="relu", solver="adam", max_iter=1000, random_state=42, learning_rate_init=0.01)
mlp_final.fit(X_treino, y_treino)
yp = mlp_final.predict(X_teste)
vp, vn, fp, fn = calc_matriz_confusao(y_teste, yp)
metricas_mlp = exibir("MLP {} - TESTE FINAL".format(melhor_cfg), vp, vn, fp, fn)



--- LOO (50 amostras originais) para MLP ---
LOO (5,) - Acuracia: 0.9600
LOO (10,) - Acuracia: 0.9800
LOO (15,) - Acuracia: 1.0000
LOO (20,) - Acuracia: 1.0000
LOO (5, 3) - Acuracia: 1.0000
LOO (10, 5) - Acuracia: 1.0000

Melhor arquitetura via LOO: (15,) (Acuracia LOO: 1.0000)

--- Treinando MLP final com todas as 400 amostras (balanceadas) ---
=== MLP (15,) - TESTE FINAL ===
           Prev POS  Prev NEG
Real POS          48         2
Real NEG           1        29

Acuracia: 0.9625
Precisao: 0.9796
Recall: 0.9600
Especificidade: 0.9667
F1-Score: 0.9697


### 7 & 8. Analise Comparativa dos Resultados


In [22]:
print("=" * 80)
print(" " * 25 + "TABELA COMPARATIVA")
print("=" * 80)
print("{:<30} {:>10} {:>10} {:>10} {:>10} {:>10}".format(
    "Modelo", "Acuracia", "Precisao", "Recall", "Especif.", "F1"))
print("-" * 80)

modelos = [
    ("Baseline (teste)", metricas_baseline),
    ("KNN k={} (teste)".format(melhor_k), metricas_knn),
    ("C4.5 prof={} (teste)".format(melhor_prof), metricas_arvore),
    ("MLP {} (teste)".format(melhor_cfg), metricas_mlp)
]

for nome, met in modelos:
    print("{:<30} {:>10.4f} {:>10.4f} {:>10.4f} {:>10.4f} {:>10.4f}".format(
        nome, met["Acuracia"], met["Precisao"], met["Recall"], met["Especificidade"], met["F1-Score"]))

print("-" * 80)
print("\nHiperparametros selecionados via validacao cruzada na base de treinamento ORIGINAL:")
print("  - KNN: k = {} (LOO)".format(melhor_k))
print("  - C4.5: profundidade = {} (10-fold CV)".format(melhor_prof))
print("  - MLP: arquitetura = {} (LOO com 50 amostras)".format(melhor_cfg))
print("\nModelo final treinado com dados balanceados (oversampling).")
print("\nAvaliacao final na base de teste:")
print("  - Todos os modelos superam o baseline.")
print("  - A validacao cruzada foi feita sobre dados ORIGINAIS sem duplicatas.")
print("  - O oversampling foi aplicado APOS a limpeza dos dados e APENAS para o treino final.")


                         TABELA COMPARATIVA
Modelo                           Acuracia   Precisao     Recall   Especif.         F1
--------------------------------------------------------------------------------
Baseline (teste)                   0.6250     0.6250     1.0000     0.0000     0.7692
KNN k=5 (teste)                    0.9625     1.0000     0.9400     1.0000     0.9691
C4.5 prof=9 (teste)                0.9750     1.0000     0.9600     1.0000     0.9796
MLP (15,) (teste)                  0.9625     0.9796     0.9600     0.9667     0.9697
--------------------------------------------------------------------------------

Hiperparametros selecionados via validacao cruzada na base de treinamento ORIGINAL:
  - KNN: k = 5 (LOO)
  - C4.5: profundidade = 9 (10-fold CV)
  - MLP: arquitetura = (15,) (LOO com 50 amostras)

Modelo final treinado com dados balanceados (oversampling).

Avaliacao final na base de teste:
  - Todos os modelos superam o baseline.
  - A validacao cruzada foi fe